In [ ]:
!pip install -q sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 61.3 MB/s eta 0:00:00


In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

model = SentenceTransformer('all-MiniLM-L6-v2')
print("model hazır")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

model hazır


In [ ]:
list_a = [
    "How do I train a model?",
    "Will I need an umbrella tomorrow?",
    "My laptop keeps shutting down on its own.",
    "Where can I buy cheap running shoes?",
    "I want to meet up with my friend this weekend.",
    "How much does rent cost in Istanbul?",
]

list_b = [
    "Options for training a model.",
    "Heavy showers are expected across the region on Tuesday.",
    "The machine powers off unexpectedly during use.",
    "Affordable sneakers are available at the outlet store.",
    "Planning to hang out with a buddy on Saturday.",
    "Average monthly rent prices in Istanbul rose this year.",
    "The recipe calls for two cups of flour.",
    "Solar panels reduce electricity bills over time.",
    "He finished reading the novel last night.",
]

print(len(list_a), len(list_b))

6 9


In [ ]:
vectors = model.encode(list_b, normalize_embeddings=True)
print(vectors.shape)

index = faiss.IndexFlatIP(vectors.shape[1])
index.add(vectors)
print("indekste kaç vektör var:", index.ntotal)

(9, 384)
indekste kaç vektör var: 9


In [ ]:
query_vectors = model.encode(list_a, normalize_embeddings=True)
scores, indices = index.search(query_vectors, k=1)

for i, query in enumerate(list_a):
    match_idx = indices[i][0]
    print(f"\nSORGU : {query}")
    print(f"EŞLEŞME: {list_b[match_idx]}")
    print(f"SKOR  : {scores[i][0]:.4f}")


SORGU : How do I train a model?
EŞLEŞME: Options for training a model.
SKOR  : 0.8048

SORGU : Will I need an umbrella tomorrow?
EŞLEŞME: Planning to hang out with a buddy on Saturday.
SKOR  : 0.3220

SORGU : My laptop keeps shutting down on its own.
EŞLEŞME: The machine powers off unexpectedly during use.
SKOR  : 0.4033

SORGU : Where can I buy cheap running shoes?
EŞLEŞME: Affordable sneakers are available at the outlet store.
SKOR  : 0.6748

SORGU : I want to meet up with my friend this weekend.
EŞLEŞME: Planning to hang out with a buddy on Saturday.
SKOR  : 0.7055

SORGU : How much does rent cost in Istanbul?
EŞLEŞME: Average monthly rent prices in Istanbul rose this year.
SKOR  : 0.7838


In [ ]:
no_match_query = ["What is the boiling point of water?"]
qv = model.encode(no_match_query, normalize_embeddings=True)
s, i = index.search(qv, k=3)

print(f"SORGU: {no_match_query[0]}\n")
for rank in range(3):
    print(f"{rank+1}. {list_b[i[0][rank]]}  →  {s[0][rank]:.4f}")

SORGU: What is the boiling point of water?

1. The recipe calls for two cups of flour.  →  0.1940
2. Heavy showers are expected across the region on Tuesday.  →  0.0835
3. Planning to hang out with a buddy on Saturday.  →  0.0724


In [ ]:
pairs = [
    ("The meeting starts at 5pm.",
     "The meeting was cancelled at 5pm."),

    ("Which team do you support?",
     "Did my team win the other day?"),

    ("My car broke down and I took it to the mechanic.",
     "The vehicle was in the workshop all week."),
]

for s1, s2 in pairs:
    v = model.encode([s1, s2], normalize_embeddings=True)
    print(f"{float(np.dot(v[0], v[1])):.4f}  |  {s1}  <->  {s2}")

0.6901  |  The meeting starts at 5pm.  <->  The meeting was cancelled at 5pm.
0.2862  |  Which team do you support?  <->  Did my team win the other day?
0.3811  |  My car broke down and I took it to the mechanic.  <->  The vehicle was in the workshop all week.


In [2]:
tr_pairs = [
    ("Arabam bozuldu, tamirciye götürdüm.",
     "Aracım bütün hafta serviste kaldı."),
    ("My car broke down and I took it to the mechanic.",
     "The vehicle was in the workshop all week."),
]

for s1, s2 in tr_pairs:
    v = model.encode([s1, s2], normalize_embeddings=True)
    print(f"{float(np.dot(v[0], v[1])):.4f}  |  {s1[:40]}...")

NameError: name 'model' is not defined

## Gözlemler

### Tahmin vs gerçek

Kodu çalıştırmadan önce her çift için modelin bulup bulamayacağını tahmin
ettim. 6 tahminden 4'ü tuttu. Model 6 sorgudan 5'ini doğru eşleştirdi.

| # | Çift | Tahminim | Gerçek | Skor |
|---|------|----------|--------|------|
| 1 | model eğitme (ortak kelimeli) | bulur | bulur | 0.8048 |
| 2 | umbrella ↔ showers | bulamaz | **bulamadı** | 0.3220 |
| 3 | laptop ↔ machine powers off | bulamaz | buldu | 0.4033 |
| 4 | cheap running shoes ↔ affordable sneakers | bulur | buldu | 0.6748 |
| 5 | friend ↔ buddy | bulur | buldu | 0.7055 |
| 6 | rent Istanbul (ortak kelimeli) | bulamaz | buldu | 0.7838 |

### Tek başarısızlık: umbrella

"Will I need an umbrella tomorrow?" sorgusuna model hava durumu cümlesini
değil, "Planning to hang out with a buddy on Saturday" cümlesini getirdi.

Sebebini düşününce anladım: şemsiye ile sağanak arasındaki bağ anlamsal
benzerlik değil, dünya bilgisi. "Şemsiye gerekir mi" demek "yağmur yağacak mı"
demek, ama bunu bilmek için bir çıkarım yapmak gerekiyor. Encoder çıkarım
yapmıyor, sadece cümlelerin benzer bağlamlarda geçip geçmediğine bakıyor.
Buddy cümlesini seçmesinin sebebi muhtemelen ikisinde de gelecek zamanlı bir
plan olması ("tomorrow" / "Saturday") — konuya değil, cümlenin şekline
takılmış.

Çıkarım: anlam yakınlığı ile mantıksal ilişki aynı şey değil.

### Skor eşiği

Üç seviye ölçtüm:
- En düşük DOĞRU eşleşme (laptop): 0.4033
- YANLIŞ eşleşme (umbrella): 0.3220
- HİÇ karşılığı olmayan sorgu ("boiling point of water"): 0.1940

Yani 0.36 civarı bir eşik hem alakasız sorguyu hem yanlış eşleşmeyi reddeder,
beş doğru eşleşmeyi kabul ederdi. Eşik mümkün ama payı ince — doğru ile yanlış
arasında sadece 0.08 var. 9 cümlelik bir oyuncakta bu kadar; binlerce adayda
skorlar birbirine yaklaşır ve bu boşluk kapanır.

Ayrıca fark ettim ki k=1 dediğim için FAISS her sorguya MUTLAKA bir cevap
döndürüyor. "Eşleşme yok" diye bir seçenek sunmuyor. Eşiği ben koymazsam
sistem her zaman bir şey bulacak, tamamen alakasız olsa bile.

### En çarpıcı bulgu: zıt anlam, aynı anlamdan yüksek skor aldı

| Skor | Çift | Anlam ilişkisi |
|------|------|----------------|
| 0.6901 | "The meeting starts at 5pm" ↔ "The meeting was cancelled at 5pm" | ZIT |
| 0.3811 | "My car broke down..." ↔ "The vehicle was in the workshop..." | AYNI |
| 0.2862 | "Which team do you support?" ↔ "Did my team win?" | FARKLI |

Zıt anlamlı çift, aynı anlamlı çiftin neredeyse iki katı skor aldı.

Sebebi: encoder cümlenin NEYLE İLGİLİ olduğunu yakalıyor, NE İDDİA ETTİĞİNİ
değil. İki toplantı cümlesi de aynı toplantı, aynı saat, neredeyse aynı
kelimeler. "Biri oluyor, diğeri olmuyor" ayrımı tek bir kelimede saklı
(cancelled) ve 384 boyutlu vektörde o tek kelimenin ağırlığı geri kalan
örtüşmenin yanında küçük kalıyor. Bunun olumsuzluk (negation) problemi diye
bilinen bir zaaf olduğunu öğrendim — "bu ilaç güvenlidir" ile "bu ilaç güvenli
değildir" de yüksek benzerlik alır.

Takım çiftinin düşük çıkması da öğreticiydi: orada gerçekten farklı iki şey
soruluyor ve ortak kelime olmasına rağmen cümlelerin geri kalanı ayrışıyor.
Yani model salt kelime saymıyor; toplantı örneğinde skorun yüksek olmasının
sebebi cümlenin neredeyse tamamının aynı olması.

Bir de kendi tasarladığım "zor paraphrase" (araba/servis) sadece 0.3811 aldı —
alakasız sorgunun 0.1940'ından iyi ama laptop eşleşmesinin 0.4033'üne bile
ulaşamadı. Gerçek paraphrase modeli ciddi şekilde zorluyor.

### Log sorusu: benzerlik skorları gerçek anlamı mı, ortak kelimeleri mi yansıtıyor?

İkisinin karışımını, ve örtüşme yüksek olduğunda anlam geri planda kalıyor.
Ortak kelimesi sıfır olan gerçek paraphrase'ları bulabiliyor (laptop/machine,
shoes/sneakers, friend/buddy) — yani salt kelime eşleştirme yapmıyor. Ama
kelime örtüşmesi çok yüksek olduğunda anlam farkını kaçırıyor (toplantı
örneği), ve çıkarım gerektiren ilişkileri hiç kuramıyor (umbrella örneği).